# Phase 4 — Customer Feature Engineering & Churn Target Formulation

**Purpose.** Transform the cleaned transaction grain into a validated, customer-level analytical matrix for machine-learning and business intelligence use. This notebook produces one record per registered customer and documents every transformation from source to output.

## Deliverables

| Deliverable | Location | Purpose |
|---|---|---|
| Input transactions | `data/processed/registered_transactions_clean.parquet` | Clean, registered transactional history from Phase 2 |
| Feature matrix | `data/processed/customer_features_matrix.parquet` | Final tabular dataset for modeling and Power BI |
| Validation figures | `reports/figures/feature_engineering/` | Reproducible SVG diagnostics; not source data |

> **Kernel requirement:** select `churn_env` in VS Code, then choose **Restart Kernel and Run All**. Do not use `base (Python 3.13.13)` for this project.

## Pipeline lineage

`clean registered transactions` → `customer RFM aggregation` → `basket metrics` → `churn labeling` → `validation and diagnostics` → `feature matrix export`

## Run order

Run all cells from top to bottom. The notebook validates its input contract before aggregation and writes the final Parquet atomically, so a failed run cannot leave a partial output file.

## Modeling contract and leakage control

The current business label is `Churn_Target = 1` when a customer has been inactive for more than 90 days at the snapshot date. Since `Recency_Days` defines that label, it is retained for reporting but **excluded from `MODEL_FEATURE_COLUMNS`**. Including it as a predictor would be target leakage and would make model evaluation misleadingly high.

The exported matrix is intentionally stored in `data/processed`, not `data/interim`: it is a complete, validated, reusable data product. `interim` is only appropriate for temporary transformations that are not ready for downstream use.

In [ ]:
# 1. Runtime configuration and project paths — run first in a fresh kernel.
import os
import sys
from pathlib import Path

# Windows guards must be set before numerical packages are imported.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('FOR_DISABLE_CONSOLE_CTRL_HANDLER', '1')

import numpy as np
import pandas as pd
from IPython.display import SVG, display

if sys.version_info >= (3, 13):
    print('WARNING: Python 3.13 is unsupported here. Select churn_env, restart, and run all cells again.')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
INPUT_PATH = PROCESSED_DATA_DIR / 'registered_transactions_clean.parquet'
OUTPUT_PATH = PROCESSED_DATA_DIR / 'customer_features_matrix.parquet'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'feature_engineering'

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f'Input parquet was not found: {INPUT_PATH}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Active Python: {sys.executable}')

## 1. Input contract

The source must contain registered customers only and the six columns below. The checks make failures explicit before any business logic is applied: `Customer ID`, `InvoiceDate`, `Invoice`, `totalprice`, `Quantity`, and `StockCode`.

In [ ]:
# 2. Load and validate the clean registered transactions.
REQUIRED_COLUMNS = {'Customer ID', 'InvoiceDate', 'Invoice', 'totalprice', 'Quantity', 'StockCode'}
df_reg = pd.read_parquet(INPUT_PATH, engine='pyarrow')
missing_columns = REQUIRED_COLUMNS.difference(df_reg.columns)
if missing_columns:
    raise ValueError(f'Input is missing required columns: {sorted(missing_columns)}')
if df_reg.empty:
    raise ValueError('Input parquet is empty; feature engineering cannot continue.')

df_reg = df_reg.copy()
df_reg['InvoiceDate'] = pd.to_datetime(df_reg['InvoiceDate'], errors='raise')
if df_reg['Customer ID'].isna().any():
    raise ValueError('Registered input contains null Customer ID values.')
for column in ('totalprice', 'Quantity'):
    df_reg[column] = pd.to_numeric(df_reg[column], errors='raise')

snapshot_date = df_reg['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Loaded {len(df_reg):,} transaction rows for {df_reg["Customer ID"].nunique():,} customers.')
print(f'Evaluation snapshot: {snapshot_date:%Y-%m-%d}')

## 2. Customer-level RFM and basket features

The aggregation produces exactly one row per `Customer_ID`. `Recency_Days` measures time since the latest invoice; frequency counts distinct invoices; monetary and unit measures are cumulative. The two ratio features describe average basket value and average basket volume.

In [ ]:
# 3. Vectorized customer-level RFM and basket feature engineering.
customer_df = (
    df_reg.groupby('Customer ID', as_index=False, sort=True)
    .agg(
        Last_Invoice=('InvoiceDate', 'max'),
        Frequency_Orders=('Invoice', 'nunique'),
        Monetary_Spend=('totalprice', 'sum'),
        Total_Units=('Quantity', 'sum'),
        Unique_Products=('StockCode', 'nunique'),
    )
    .rename(columns={'Customer ID': 'Customer_ID'})
)
customer_df['Recency_Days'] = (snapshot_date - customer_df.pop('Last_Invoice')).dt.days.astype('int32')
customer_df['Avg_Order_Value'] = customer_df['Monetary_Spend'].div(customer_df['Frequency_Orders']).round(2)
customer_df['Avg_Units_Per_Order'] = customer_df['Total_Units'].div(customer_df['Frequency_Orders']).round(2)

if (customer_df['Frequency_Orders'] <= 0).any():
    raise ValueError('Frequency_Orders must be positive for every customer.')
if not np.isfinite(customer_df.select_dtypes(include='number').to_numpy()).all():
    raise ValueError('Feature matrix contains NaN or infinite numeric values.')
if customer_df['Customer_ID'].duplicated().any():
    raise ValueError('Customer aggregation produced duplicate Customer_ID values.')

print(f'Created customer matrix: {customer_df.shape[0]:,} rows × {customer_df.shape[1]} columns')
display(customer_df.head())

## 3. Churn target and multicollinearity audit

The 90-day threshold is a transparent business rule: it yields a binary label for the current snapshot. Correlations and VIF then identify redundant features before modeling. VIF is calculated using a small pure-Python matrix inverse; this deliberately avoids the NumPy-LAPACK call that was crashing the Windows kernel.

In [ ]:
# 4. Churn target and stable multicollinearity audit.
CHURN_THRESHOLD_DAYS = 90
customer_df['Churn_Target'] = (customer_df['Recency_Days'] > CHURN_THRESHOLD_DAYS).astype('int8')
FEATURE_COLUMNS = [
    'Frequency_Orders', 'Monetary_Spend', 'Total_Units', 'Unique_Products',
    'Avg_Order_Value', 'Avg_Units_Per_Order',
]
MODEL_FEATURE_COLUMNS = FEATURE_COLUMNS.copy()

churn_counts = customer_df['Churn_Target'].value_counts().reindex([0, 1], fill_value=0)
churn_pct = customer_df['Churn_Target'].value_counts(normalize=True).reindex([0, 1], fill_value=0).mul(100)
print(f'Retained (0): {churn_counts[0]:,} ({churn_pct[0]:.2f}%)')
print(f'Churned  (1): {churn_counts[1]:,} ({churn_pct[1]:.2f}%)')

correlation_matrix = customer_df[FEATURE_COLUMNS].corr(method='pearson')

def invert_small_matrix(matrix, tolerance=1e-12):
    """Gauss-Jordan inverse for a tiny VIF matrix; avoids NumPy-LAPACK crashes."""
    n = len(matrix)
    augmented = [[float(value) for value in matrix[row]] + [float(row == col) for col in range(n)] for row in range(n)]
    for column in range(n):
        pivot_row = max(range(column, n), key=lambda row: abs(augmented[row][column]))
        if abs(augmented[pivot_row][column]) < tolerance:
            raise ValueError('VIF cannot be computed: feature correlation matrix is singular.')
        augmented[column], augmented[pivot_row] = augmented[pivot_row], augmented[column]
        pivot = augmented[column][column]
        augmented[column] = [value / pivot for value in augmented[column]]
        for row in range(n):
            if row != column:
                factor = augmented[row][column]
                augmented[row] = [value - factor * pivot_value for value, pivot_value in zip(augmented[row], augmented[column])]
    return [row[n:] for row in augmented]

inverse_correlation = invert_small_matrix(correlation_matrix.to_numpy(dtype=float).tolist())
vif_results = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'VIF': [inverse_correlation[index][index] for index in range(len(FEATURE_COLUMNS))],
}).sort_values('VIF', ascending=False)

print('Correlation matrix:')
display(correlation_matrix.round(3))
print('VIF audit:')
display(vif_results.round(2))

## 4. Visual validation

The charts below are professional, portable **SVG** diagnostics. They answer two essential questions: whether the classes are balanced and where customer inactivity sits relative to the 90-day decision boundary. They are created without Matplotlib because that library is the component that currently crashes this Windows kernel. SVG is a proper vector graphics format: sharp in notebooks, Power BI documentation, and reports.

The files are placed under `reports/figures/feature_engineering/`, keeping generated visuals separate from the data products in `data/processed/`.

In [ ]:
# 5. Generate safe vector diagnostics without Matplotlib.
from html import escape

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_svg(path, width, height, body):
    header = f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">'
    style = '<style>text{font-family:Segoe UI,Arial,sans-serif;fill:#172b4d}.title{font-size:20px;font-weight:700}.label{font-size:13px}.value{font-size:14px;font-weight:700}</style>'
    path.write_text(header + style + '<rect width="100%" height="100%" fill="white"/>' + body + '</svg>', encoding='utf-8')

# Class-balance bar chart.
bar_width, bar_height, left, baseline = 720, 420, 90, 340
maximum = max(int(churn_counts.max()), 1)
labels = ['Retained (0)', 'Churned (1)']
colors = ['#20a464', '#d84a4a']
body = ['<text x="90" y="45" class="title">Churn target distribution</text>', '<text x="90" y="70" class="label">Customer counts at the 90-day inactivity threshold</text>', f'<line x1="{left}" y1="{baseline}" x2="650" y2="{baseline}" stroke="#9aa5b1"/>']
for index, (label, color) in enumerate(zip(labels, colors)):
    count = int(churn_counts.iloc[index])
    height = round(count / maximum * 230)
    x = 180 + index * 240
    y = baseline - height
    body.extend([f'<rect x="{x}" y="{y}" width="120" height="{height}" rx="6" fill="{color}"/>', f'<text x="{x + 60}" y="{y - 12}" text-anchor="middle" class="value">{count:,} ({churn_pct.iloc[index]:.1f}%)</text>', f'<text x="{x + 60}" y="370" text-anchor="middle" class="label">{escape(label)}</text>'])
class_chart_path = FIGURES_DIR / 'churn_class_distribution.svg'
save_svg(class_chart_path, bar_width, bar_height, ''.join(body))

# Recency histogram with threshold line.
values = customer_df['Recency_Days'].astype(int).tolist()
bin_count, value_min, value_max = 35, min(values), max(values)
bin_size = max((value_max - value_min + 1) / bin_count, 1)
histogram = [0] * bin_count
for value in values:
    histogram[min(int((value - value_min) / bin_size), bin_count - 1)] += 1
maximum_bin = max(histogram)
plot_left, plot_right, plot_top, plot_bottom = 75, 680, 100, 340
body = ['<text x="75" y="45" class="title">Customer recency distribution</text>', '<text x="75" y="70" class="label">Days since the latest completed invoice</text>', f'<line x1="{plot_left}" y1="{plot_bottom}" x2="{plot_right}" y2="{plot_bottom}" stroke="#9aa5b1"/>']
bar_pixel_width = (plot_right - plot_left) / bin_count
for index, count in enumerate(histogram):
    height = 0 if maximum_bin == 0 else count / maximum_bin * (plot_bottom - plot_top)
    x = plot_left + index * bar_pixel_width
    body.append(f'<rect x="{x:.1f}" y="{plot_bottom - height:.1f}" width="{max(bar_pixel_width - 1, 1):.1f}" height="{height:.1f}" fill="#247ba0"/>')
threshold_x = plot_left + (CHURN_THRESHOLD_DAYS - value_min) / (value_max - value_min) * (plot_right - plot_left)
body.extend([f'<line x1="{threshold_x:.1f}" y1="{plot_top}" x2="{threshold_x:.1f}" y2="{plot_bottom}" stroke="#d62828" stroke-width="3" stroke-dasharray="7 5"/>', f'<text x="{min(threshold_x + 8, 570):.1f}" y="120" class="value" fill="#d62828">90-day cut-off</text>', f'<text x="{plot_left}" y="375" class="label">{value_min} days</text>', f'<text x="{plot_right}" y="375" text-anchor="end" class="label">{value_max} days</text>'])
recency_chart_path = FIGURES_DIR / 'recency_distribution.svg'
save_svg(recency_chart_path, 760, 420, ''.join(body))

display(SVG(filename=str(class_chart_path)))
display(SVG(filename=str(recency_chart_path)))
print(f'Figures saved to: {FIGURES_DIR}')

## 5. Export and hand-off

The final cell writes the validated matrix to `data/processed/customer_features_matrix.parquet`. The temporary-file swap is atomic on the local filesystem: the existing output is replaced only after the new Parquet has been written successfully. The displayed predictor list is the contract for the next modeling notebook.

In [ ]:
# 6. Atomically export the validated feature matrix.
EXPECTED_COLUMNS = [
    'Customer_ID', 'Frequency_Orders', 'Monetary_Spend', 'Total_Units',
    'Unique_Products', 'Recency_Days', 'Avg_Order_Value',
    'Avg_Units_Per_Order', 'Churn_Target',
]
customer_df = customer_df[EXPECTED_COLUMNS].copy()
temp_output_path = OUTPUT_PATH.with_suffix('.tmp.parquet')
customer_df.to_parquet(temp_output_path, index=False, engine='pyarrow')
temp_output_path.replace(OUTPUT_PATH)

print('Phase 4 completed successfully.')
print(f'Exported matrix: {OUTPUT_PATH.resolve()}')
print(f'Final matrix: {customer_df.shape[0]:,} rows × {customer_df.shape[1]} columns')
print(f'Model predictors: {MODEL_FEATURE_COLUMNS}')
display(customer_df.head())